In [ ]:
from datasets import load_dataset, load_from_disk
import librosa
import soundfile as sf
import numpy as np

def check_audio_quality(dataset, sample_size=100):
    """Check for common audio issues."""
    import random
    samples = random.sample(range(len(dataset)), min(sample_size, len(dataset)))

    issues = {
        'silent': [],
        'clipping': [],
        'low_volume': []
    }

    for idx in samples:
        audio = dataset[idx]['audio']['array']

        # Check for silence
        if abs(audio).max() < 0.01:
            issues['silent'].append(idx)

        # Check for clipping
        if abs(audio).max() > 0.95:
            issues['clipping'].append(idx)

        # Check for low volume
        rms = np.sqrt(np.mean(audio**2))
        if rms < 0.005:
            issues['low_volume'].append(idx)

    print(f"Silent samples: {len(issues['silent'])} / {sample_size}")
    print(f"Clipping samples: {len(issues['clipping'])} / {sample_size}")
    print(f"Low volume samples: {len(issues['low_volume'])} / {sample_size}")

    return issues

def check_transcription_quality(dataset):
    """Check for common transcription issues."""
    issues = {
        'empty': [],
        'too_short': [],
        'too_long': [],
        'invalid_chars': []
    }

    for idx, example in enumerate(dataset):
        text = example['text']

        # Empty transcription
        if not text or len(text.strip()) == 0:
            issues['empty'].append(idx)

        # Too short (< 3 characters)
        elif len(text) < 3:
            issues['too_short'].append(idx)

        # Too long (> 500 characters)
        elif len(text) > 500:
            issues['too_long'].append(idx)

        # Check for invalid characters (optional)
        # if any(ord(c) > 127 for c in text):
        #     issues['invalid_chars'].append(idx)

    print(f"Empty transcriptions: {len(issues['empty'])}")
    print(f"Too short: {len(issues['too_short'])}")
    print(f"Too long: {len(issues['too_long'])}")

    return issues

def compute_dataset_stats(dataset):
    """Compute statistics about the dataset."""
    durations = []
    text_lengths = []

    for example in dataset:
        audio = example['audio']
        duration = len(audio['array']) / audio['sampling_rate']
        durations.append(duration)

        text_length = len(example['text'])
        text_lengths.append(text_length)

    print(f"Number of samples: {len(dataset)}")
    print(f"\nAudio Duration Statistics:")
    print(f"  Mean: {np.mean(durations):.2f}s")
    print(f"  Median: {np.median(durations):.2f}s")
    print(f"  Min: {np.min(durations):.2f}s")
    print(f"  Max: {np.max(durations):.2f}s")
    print(f"  Total: {np.sum(durations)/3600:.2f} hours")

    print(f"\nTranscription Length Statistics:")
    print(f"  Mean: {np.mean(text_lengths):.1f} chars")
    print(f"  Median: {np.median(text_lengths):.1f} chars")
    print(f"  Min: {np.min(text_lengths)} chars")
    print(f"  Max: {np.max(text_lengths)} chars")

In [5]:
dataset_path = "/scratch/cjh9fw/finetune-moonshine-asr"

print(f"\nLoading local dataset from: {dataset_path}")
dataset = load_from_disk(dataset_path)

In [ ]:
compute_dataset_stats(dataset['train'])

# issues = check_audio_quality(dataset['train'])
# issues = check_transcription_quality(dataset['train'])


In [ ]:
print(dataset['train'][-1]['audio'])
print(dataset['train'][-1]['text'])

audio = dataset['train'][-1]['audio']


print(type(audio))
print("sampling_rate:", audio["sampling_rate"])

samples = audio.get_all_samples()
print("sample_rate:", samples.sample_rate)
print("shape:", samples.data.shape)
print("duration_sec:", samples.data.shape[-1] / samples.sample_rate)
print("path:", audio.metadata.path)
print(samples.keys())





Train dataset num samples:  724756
Test dataset num samples:  80516
I want that! Cancel the Chinese food! I understand your excitement. That might be our Grilled Tom Yum Jumbo Prawns with Garlic Rice. Unfortunately, I cannot modify an order once it has been confirmed. Aiyo, seriously? Okay, okay, my fault. Just leave it then. Thanks.
<class 'datasets.features._torchcodec.AudioDecoder'>
sampling_rate: 16000
sample_rate: 16000
shape: torch.Size([1, 343040])
duration_sec: 21.44
path: recording.wav
Silent samples: 0 / 100
Clipping samples: 0 / 100
Low volume samples: 0 / 100
